# Associative Recall: Spiking Self-Attention vs. Vanilla Transformer

Trains two small models on a synthetic **causal** Associative Recall task (induction-head
style, as in Ba et al. 2016 / H3 / Based / Hyena) and compares them:

- **VanillaTransformer** — standard softmax multi-head self-attention (baseline).
- **SpikingTransformer** — Spikformer-style Spiking Self Attention (SSA): Q/K/V are binary
  spike tensors produced by `Linear -> LayerNorm -> LIF`, and attention is computed as
  `SN(scale * (Q @ K^T) @ V)` with no softmax, simulated over `T` discrete time steps.

Task: the same N key-value pairs are presented for R rounds (order reshuffled each round).
Trained with ordinary next-token prediction; recall accuracy is measured right after a key
token, in round >= 2, where the paired value is actually predictable from earlier context.

All the actual data/model/training code lives in `data.py` / `modules.py` / `models.py` /
`train.py` — this notebook just imports and drives it, and adds a worked example at the end.

In [8]:
import torch

from data import generate_batch, key_position_mask, seq_len, vocab_size
from models import SpikingTransformer, VanillaTransformer
from train import compute_loss_and_acc, pick_device, plot_comparison, train_one_model


## Config

In [ ]:
num_kv_pairs = 8
num_rounds = 2
num_keys = 32
num_values = 32

dim = 64
depth = 2
num_heads = 4
T = 10  # spiking simulation time steps

batch_size = 64
eval_batch_size = 512
steps = 3000
eval_every = 100
lr = 1e-3
seed = 0

torch.manual_seed(seed)
device = pick_device()

V = vocab_size(num_keys, num_values)
L = seq_len(num_kv_pairs, num_rounds)
mask = key_position_mask(num_kv_pairs, num_rounds, device=device)
print(f"device={device}, vocab_size={V}, seq_len={L}, num_kv_pairs={num_kv_pairs}, num_rounds={num_rounds}")


device=mps, vocab_size=64, seq_len=32, num_kv_pairs=8, num_rounds=2


## Models

In [10]:
vanilla = VanillaTransformer(V, dim=dim, depth=depth, num_heads=num_heads, max_len=L).to(device)
spiking = SpikingTransformer(V, dim=dim, depth=depth, num_heads=num_heads, T=T, max_len=L).to(device)

print(f"VanillaTransformer params: {sum(p.numel() for p in vanilla.parameters()):,}")
print(f"SpikingTransformer params: {sum(p.numel() for p in spiking.parameters()):,}")


VanillaTransformer params: 110,272
SpikingTransformer params: 111,808


## Training

Uses `train_one_model` from `train.py` for both models.

In [11]:
hist_vanilla = train_one_model(
    "vanilla", vanilla, mask, device,
    num_kv_pairs=num_kv_pairs, num_rounds=num_rounds,
    num_keys=num_keys, num_values=num_values,
    batch_size=batch_size, eval_batch_size=eval_batch_size,
    steps=steps, eval_every=eval_every, lr=lr,
)


[vanilla] step   100 | eval loss 3.4734 | eval acc   4.7% | 1.8s
[vanilla] step   200 | eval loss 3.0833 | eval acc  14.3% | 3.2s
[vanilla] step   300 | eval loss 2.7527 | eval acc  16.8% | 4.6s
[vanilla] step   400 | eval loss 2.5138 | eval acc  18.4% | 6.0s
[vanilla] step   500 | eval loss 2.3116 | eval acc  21.4% | 7.3s
[vanilla] step   600 | eval loss 2.0696 | eval acc  29.6% | 8.7s
[vanilla] step   700 | eval loss 0.9176 | eval acc  72.2% | 10.1s
[vanilla] step   800 | eval loss 0.0820 | eval acc  97.7% | 11.4s
[vanilla] step   900 | eval loss 0.0048 | eval acc 100.0% | 12.8s
[vanilla] step  1000 | eval loss 0.0024 | eval acc 100.0% | 14.1s
[vanilla] step  1100 | eval loss 0.0016 | eval acc 100.0% | 15.4s
[vanilla] step  1200 | eval loss 0.0011 | eval acc 100.0% | 16.8s
[vanilla] step  1300 | eval loss 0.0008 | eval acc 100.0% | 18.1s
[vanilla] step  1400 | eval loss 0.0007 | eval acc 100.0% | 19.4s
[vanilla] step  1500 | eval loss 0.0006 | eval acc 100.0% | 20.7s
[vanilla] step  

In [12]:
hist_spiking = train_one_model(
    "spiking", spiking, mask, device,
    num_kv_pairs=num_kv_pairs, num_rounds=num_rounds,
    num_keys=num_keys, num_values=num_values,
    batch_size=batch_size, eval_batch_size=eval_batch_size,
    steps=steps, eval_every=eval_every, lr=lr,
)


[spiking] step   100 | eval loss 3.5417 | eval acc   3.9% | 8.0s
[spiking] step   200 | eval loss 3.3720 | eval acc   8.4% | 15.8s
[spiking] step   300 | eval loss 3.1834 | eval acc  12.1% | 23.8s
[spiking] step   400 | eval loss 2.9150 | eval acc  17.5% | 31.5s
[spiking] step   500 | eval loss 2.7257 | eval acc  19.3% | 39.3s
[spiking] step   600 | eval loss 2.6460 | eval acc  19.7% | 47.0s
[spiking] step   700 | eval loss 2.5724 | eval acc  20.2% | 54.9s
[spiking] step   800 | eval loss 2.5233 | eval acc  19.8% | 62.8s
[spiking] step   900 | eval loss 2.4541 | eval acc  21.0% | 70.5s
[spiking] step  1000 | eval loss 2.4186 | eval acc  21.3% | 78.3s
[spiking] step  1100 | eval loss 2.3929 | eval acc  20.1% | 86.2s
[spiking] step  1200 | eval loss 2.3671 | eval acc  20.1% | 94.4s
[spiking] step  1300 | eval loss 2.3377 | eval acc  20.1% | 102.3s
[spiking] step  1400 | eval loss 2.2982 | eval acc  20.5% | 110.7s
[spiking] step  1500 | eval loss 2.2743 | eval acc  21.1% | 118.5s
[spiking

KeyboardInterrupt: 

## Results

In [ ]:
print("=== Final comparison ===")
print(f"VanillaTransformer  final eval acc: {hist_vanilla['acc'][-1]*100:.1f}%")
print(f"SpikingTransformer  final eval acc: {hist_spiking['acc'][-1]*100:.1f}%")


In [ ]:
fig = plot_comparison(hist_vanilla, hist_spiking)
fig


## Example test output

Pull a fresh sample and show, in a human-readable way, each round of key-value pairs and,
for every key position in round >= 2 (the positions that are actually answerable), the
ground-truth value versus what each trained model predicted.

In [13]:
seq = generate_batch(1, num_kv_pairs, num_rounds, num_keys, num_values, device=device)

vanilla.eval()
spiking.eval()
with torch.no_grad():
    pred_vanilla = vanilla(seq)[:, :-1, :].argmax(-1)[0]
    pred_spiking = spiking(seq)[:, :-1, :].argmax(-1)[0]

tokens = seq[0].cpu().tolist()
targets = seq[0, 1:].cpu().tolist()
mask_cpu = mask.cpu().tolist()

print("full sequence:", tokens)
print()
for r in range(num_rounds):
    round_tokens = tokens[r * 2 * num_kv_pairs:(r + 1) * 2 * num_kv_pairs]
    pairs = [f"{round_tokens[j]}->{round_tokens[j+1]}" for j in range(0, len(round_tokens), 2)]
    print(f"round {r + 1}: {', '.join(pairs)}")
print()

print(f"{'pos':>4} {'key':>4} {'target(value)':>14} {'vanilla':>8} {'spiking':>8}")
for t in range(len(targets)):
    if not mask_cpu[t]:
        continue
    key_tok = tokens[t]
    truth = targets[t]
    pv = pred_vanilla[t].item()
    ps = pred_spiking[t].item()
    print(f"{t:>4} {key_tok:>4} {truth:>14} {pv:>5} ({'OK' if pv == truth else 'X'}) "
          f"{ps:>5} ({'OK' if ps == truth else 'X'})")


full sequence: [15, 34, 19, 45, 28, 32, 12, 56, 1, 61, 0, 63, 17, 55, 3, 32, 19, 45, 1, 61, 12, 56, 15, 34, 28, 32, 17, 55, 3, 32, 0, 63]

round 1: 15->34, 19->45, 28->32, 12->56, 1->61, 0->63, 17->55, 3->32
round 2: 19->45, 1->61, 12->56, 15->34, 28->32, 17->55, 3->32, 0->63

 pos  key  target(value)  vanilla  spiking
  16   19             45    45 (OK)    32 (X)
  18    1             61    61 (OK)    55 (X)
  20   12             56    56 (OK)    32 (X)
  22   15             34    34 (OK)    32 (X)
  24   28             32    32 (OK)    32 (OK)
  26   17             55    55 (OK)    32 (X)
  28    3             32    32 (OK)    32 (OK)
  30    0             63    63 (OK)    32 (X)
